In [1]:
!pip install flask torch torchvision pillow matplotlib opencv-python cloudflared


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 2.6 MB/s eta 0:00:00
  Created wheel for cloudflared: filename=cloudflared-1.0.0.2-py3-none-any.whl size=2983 sha256=7e892a2b73b3a6ae5d1142df2fe335d858520e169b729012535139b93f0e6cf7
  Stored in directory: /root/.cache/pip/wheels/5b/ec/09/c3bcd3470be046ec77a9c0cb9d8bb6ceed49c831460878ab0a
Successfully built cloudflared


In [2]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!sudo mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

--2026-03-04 04:46:39--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64 [following]
--2026-03-04 04:46:39--  https://github.com/cloudflare/cloudflared/releases/download/2026.2.0/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/f9298ca8-89c8-41fe-a51f-e24cb2059878?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-04T05%3A36%3A27Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-04T0

In [2]:
!pip install pydicom

import io
import torch
import torch.nn as nn
from torchvision import models, transforms
from flask import Flask, request, render_template_string
from PIL import Image
import numpy as np
import cv2
import threading
import base64
import pydicom
from pydicom.filebase import DicomBytesIO
from google.colab import drive

# ====== Setup Flask ======
app = Flask(__name__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== Load Trained Model ======
model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # 2 classes: Normal, Pneumonia

drive.mount('/content/drive')
model.load_state_dict(torch.load('/content/drive/MyDrive/pneumonia_resnet18_best.pth',
                                 map_location=device), strict=False)
model = model.to(device)
model.eval()

# ====== Image Preprocessing ======
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ====== Grad-CAM Helper ======
def generate_gradcam(model, img_tensor, target_class):
    gradients = []
    activations = []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    def forward_hook(module, input, output):
        activations.append(output)

    last_conv = model.layer4[-1].conv2
    forward_handle = last_conv.register_forward_hook(forward_hook)
    backward_handle = last_conv.register_backward_hook(backward_hook)

    output = model(img_tensor)
    model.zero_grad()
    class_score = output[0, target_class]
    class_score.backward()

    grads = gradients[0].cpu().data.numpy()[0]
    acts = activations[0].cpu().data.numpy()[0]
    weights = np.mean(grads, axis=(1, 2))
    gradcam = np.zeros(acts.shape[1:], dtype=np.float32)
    for i, w in enumerate(weights):
        gradcam += w * acts[i, :, :]
    gradcam = np.maximum(gradcam, 0)
    gradcam = cv2.resize(gradcam, (224, 224))
    gradcam -= gradcam.min()
    gradcam /= gradcam.max()

    forward_handle.remove()
    backward_handle.remove()
    return gradcam

# ====== HTML Upload Page ======
UPLOAD_HTML = """
<!doctype html>
<html>
<head>
    <title>Chest X-Ray Prediction</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f5f5f5;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: flex-start;
            padding-top: 50px;
        }
        h2 { color: #333; }
        form {
            background: #fff;
            padding: 20px 30px;
            border-radius: 12px;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
            display: flex;
            flex-direction: column;
            align-items: center;
            margin-bottom: 30px;
        }
        input[type="file"] { margin-bottom: 15px; }
        input[type="submit"] {
            background-color: #007BFF;
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        input[type="submit"]:hover { background-color: #0056b3; }
        .result {
            text-align: center;
            background: #fff;
            padding: 20px 30px;
            border-radius: 12px;
            box-shadow: 0 4px 10px rgba(0,0,0,0.1);
        }
        .result img {
            margin-top: 15px;
            max-width: 400px;
            border-radius: 10px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.2);
        }
    </style>
</head>
<body>
    <h2>Upload a Chest X-Ray Image</h2>
    <form action="/" method="post" enctype="multipart/form-data">
        <input type="file" name="file" accept=".dcm,.jpg,.jpeg,.png" required>
        <input type="submit" value="Upload">
    </form>

    {% if prediction %}
    <div class="result">
        <h3>Prediction: {{prediction}}</h3>
        <h3>Pneumonia Probability: {{score}}%</h3>
        <img src="data:image/jpeg;base64,{{gradcam_img}}" alt="Grad-CAM">
    </div>
    {% endif %}
</body>
</html>
"""

# ====== Home Route (Handles Upload & Prediction) ======
@app.route('/', methods=['GET', 'POST'])
def home():
    prediction = None
    score = None
    gradcam_img = None

    if request.method == 'POST' and 'file' in request.files:
        file = request.files['file']
        file_bytes = file.read()
        filename = file.filename.lower()

        try:
            if filename.endswith('.dcm'):
                # ====== Handle DICOM File ======
                ds = pydicom.dcmread(DicomBytesIO(file_bytes))
                pixel_array = ds.pixel_array.astype(np.float32)
                pixel_array -= pixel_array.min()
                pixel_array /= pixel_array.max()
                pixel_array = (pixel_array * 255).astype(np.uint8)
                if len(pixel_array.shape) == 2:
                    pixel_array = np.stack([pixel_array] * 3, axis=-1)
                img = Image.fromarray(pixel_array).convert('RGB')
            else:
                # ====== Handle JPG/PNG File ======
                img = Image.open(io.BytesIO(file_bytes)).convert('RGB')

            img_tensor = transform(img).unsqueeze(0).to(device)

            # ====== Prediction ======
            with torch.no_grad():
                output = model(img_tensor)
                probs = torch.softmax(output, dim=1)
                pred_class = output.argmax(dim=1).item()
                pneumonia_score = probs[0, 1].item()

            classes = ['Normal', 'Pneumonia']
            prediction = classes[pred_class]
            score = round(pneumonia_score * 100, 2)

            # ====== Grad-CAM ======
            gradcam = generate_gradcam(model, img_tensor, pred_class)
            img_np = np.array(img.resize((224, 224)))
            heatmap = cv2.applyColorMap(np.uint8(255 * gradcam), cv2.COLORMAP_JET)
            overlay = cv2.addWeighted(img_np, 0.6, heatmap, 0.4, 0)

            _, buffer = cv2.imencode('.jpg', cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
            gradcam_img = base64.b64encode(buffer).decode('utf-8')

        except Exception as e:
            app.logger.error(f"Exception on / [POST]\n{e}", exc_info=True)

    return render_template_string(UPLOAD_HTML,
                                  prediction=prediction,
                                  score=score,
                                  gradcam_img=gradcam_img)

# ====== Run Flask in Background ======
def run_flask():
    app.run(host='0.0.0.0', port=50013)

threading.Thread(target=run_flask).start()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!cloudflared tunnel --url http://localhost:50013


2026-03-04T05:01:59Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-04T05:01:59Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-04T05:02:02Z INF +--------------------------------------------------------------------------------------------+
2026-03-04T05:02:02Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-04T05:02:02Z INF |  https://wow-routines-computing-steven.trycloudflare.c

INFO:werkzeug:127.0.0.1 - - [04/Mar/2026 05:02:18] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Mar/2026 05:02:19] "GET /favicon.ico HTTP/1.1" 404 -
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1867: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)
INFO:werkzeug:127.0.0.1 - - [04/Mar/2026 05:02:27] "POST / HTTP/1.1" 200 -


2026-03-04T05:02:41Z INF Initiating graceful shutdown due to signal interrupt ...
2026-03-04T05:02:41Z ERR failed to run the datagram handler error="context canceled" connIndex=0 event=0 ip=198.41.200.13
2026-03-04T05:02:41Z ERR failed to serve tunnel connection error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.13
2026-03-04T05:02:41Z ERR Serve tunnel error error="accept stream listener encountered a failure while serving" connIndex=0 event=0 ip=198.41.200.13
2026-03-04T05:02:41Z INF Retrying connection in up to 1s connIndex=0 event=0 ip=198.41.200.13
2026-03-04T05:02:41Z ERR Connection terminated connIndex=0
2026-03-04T05:02:41Z ERR no more connections active and exiting
2026-03-04T05:02:41Z INF Tunnel server stopped
2026-03-04T05:02:41Z INF Metrics server stopped


In [1]:
!fuser -k 50013/tcp